<a href="https://colab.research.google.com/github/jacobwechuli/mypython/blob/main/Chunking_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q llama-index llama-index-embeddings-huggingface llama-index-llms-gemini chromadb llama-index-vector-stores-chroma scikit-learn

In [3]:
# Create a folder to store our sample documents
!mkdir -p sample_docs

In [4]:
# ============================================
# STEP 2: Create Sample Documents
# ============================================
#
# WHAT WE'RE DOING: Creating 3 sample text files about AI topics.
# These will be the "documents" our AI will search through.
# WHY THIS MATTERS: In the real world, these would be pharmaceutical documents,
# research papers, or quality reports. We use simple AI text for practice.
# WHAT YOU'LL SEE: No output — the files are created silently in the sample_docs folder.
# ============================================

# Let's create some example documents
with open("sample_docs/ai_history.txt", "w") as f:
    f.write("""Artificial Intelligence (AI) has a rich history dating back to the 1950s.
The term was first coined by John McCarthy in 1956 at the Dartmouth Conference.
Early AI research focused on symbolic methods and rule-based systems.
In the 1980s, expert systems became popular but faced limitations.
The 1990s and early 2000s saw a shift towards machine learning approaches.
The deep learning revolution began around 2012 with breakthrough results in computer vision.
Today, large language models like GPT, LLaMA, and Gemini represent cutting-edge AI capabilities.
These models are trained on vast amounts of text data and can generate human-like responses.""")

with open("sample_docs/neural_networks.txt", "w") as f:
    f.write("""Neural networks are computing systems inspired by biological neural networks.
The perceptron, developed by Frank Rosenblatt in 1958, was one of the earliest neural network models.
Modern neural networks consist of layers of interconnected nodes or "neurons."
Each connection can transmit a signal from one neuron to another.
The receiving neuron processes the signal and signals downstream neurons connected to it.
Deep neural networks contain multiple hidden layers between input and output layers.
Convolutional Neural Networks (CNNs) revolutionized image processing.
Recurrent Neural Networks (RNNs) and transformers handle sequential data like text or time series.""")

with open("sample_docs/embeddings.txt", "w") as f:
    f.write("""Embeddings are dense vector representations of data in a continuous vector space.
Word embeddings map words to vectors where similar words are positioned closer together.
Popular word embedding techniques include Word2Vec, GloVe, and FastText.
Sentence embeddings capture meaning at the sentence level rather than individual words.
Models like Universal Sentence Encoder and SBERT create powerful sentence embeddings.
Document embeddings represent entire documents as fixed-length vectors.
Embeddings enable semantic search by finding documents with similar meaning, not just keyword matches.
They also power recommendation systems, clustering, and classification tasks.
Recent models like OpenAI's text-embedding-ada and Cohere's embedding models offer state-of-the-art performance.""")

In [ ]:
from llama_index.core import SimpleDirectoryReader

# Load PDF or text document
documents = SimpleDirectoryReader("sample_docs").load_data()
print(f"Loaded {len(documents)} documents.")

In [7]:
from llama_index.core.node_parser import SentenceSplitter

splitter_fixed = SentenceSplitter(chunk_size=300, chunk_overlap=0)  # No overlap
chunks_fixed = splitter_fixed.get_nodes_from_documents(documents)
print(f"Total Fixed-Length Chunks Created: {len(chunks_fixed)}")

Total Fixed-Length Chunks Created: 3


In [8]:
splitter_overlap = SentenceSplitter(chunk_size=300, chunk_overlap=50)  # 50-token overlap
chunks_overlap = splitter_overlap.get_nodes_from_documents(documents)
print(f"Total Overlapping Chunks Created: {len(chunks_overlap)}")

Total Overlapping Chunks Created: 3


In [9]:
from llama_index.core.node_parser import SemanticSplitterNodeParser
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# Semantic chunking needs an embedding model to detect topic shifts
embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
semantic_splitter = SemanticSplitterNodeParser(embed_model=embed_model)
chunks_semantic = semantic_splitter.get_nodes_from_documents(documents)
print(f"Total Semantic Chunks Created: {len(chunks_semantic)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:121: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Total Semantic Chunks Created: 6


In [ ]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# Load embedding model
embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Apply embeddings
for chunk in chunks_overlap:  # Using Overlapping Chunks for best retrieval
    chunk.embedding = embed_model.get_text_embedding(chunk.text)

print("Embeddings Generated Successfully!")


In [ ]:
from llama_index.core import VectorStoreIndex
from llama_index.llms.gemini import Gemini

# Set up Gemini as our LLM for generating answers
llm = Gemini(model="models/gemini-2.0-flash")

# Create an index with our embeddings
index = VectorStoreIndex.from_documents(documents, embed_model=embed_model)

# Set up query engine with Gemini
query_engine = index.as_query_engine(llm=llm, similarity_top_k=2)

# Test a retrieval query
response = query_engine.query("What is the document about?")
print(response)